In [2]:
%%configure
{
    "defaultLakehouse": {
        "name": "lh_oyster_path"
    }
}

StatementMeta(, a4167e5c-85e8-4c01-a3fa-a0861fb129ea, -1, Finished, Available, Finished, True)

In [11]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F




StatementMeta(, 2bc70166-4f34-4162-ae65-e5a79a21336c, 13, Finished, Available, Finished, False)

In [12]:
# Define the relative path using wildcards to capture all date folders
# This targets all 'Customer' CSV files across any date folder inside 'oyster'
p_root_path = f"Files/landing/oyster/"
p_in_table_name = "Customer"

StatementMeta(, 2bc70166-4f34-4162-ae65-e5a79a21336c, 14, Finished, Available, Finished, False)

In [13]:
target_table = f"dbo.ld_oys_{p_in_table_name}".lower()

StatementMeta(, 2bc70166-4f34-4162-ae65-e5a79a21336c, 15, Finished, Available, Finished, False)

In [14]:

# Build the recursive glob pattern
pattern = f"{p_root_path.rstrip('/')}/**/{p_in_table_name}_*.csv"

# Extract only the file paths (keys) to avoid loading file contents into driver memory
source_files = spark.sparkContext.binaryFiles(pattern).keys().collect()

if not source_files:
    raise FileNotFoundError(f"No raw files match {pattern}")

# Sort paths chronologically by the yyyymmddhhmmss folder name and pick the last one
# Note: Extracting the parent folder name (-2) ensures sorting is strictly based on the folder
latest_file_path = sorted(source_files, key=lambda path: path.split("/")[-2])[-1]

print(f"Matched {len(source_files)} file(s).")
print(f"Latest file selected: {latest_file_path}")


StatementMeta(, 2bc70166-4f34-4162-ae65-e5a79a21336c, 16, Finished, Available, Finished, False)

Matched 1 file(s).
Latest file selected: abfss://5e32bf37-c2be-4676-94c7-97213238ed30@onelake.dfs.fabric.microsoft.com/f27d086e-be7b-4e40-8e62-ea4d210b4538/Files/landing/oyster/20261109142245/Customer_20261109T142245.csv


In [15]:
# Read only the latest CSV file
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(latest_file_path)
)

# Clean up the file name (removes '?version=...' suffix) and add metadata columns
raw_file_name = F.element_at(F.split(F.input_file_name(), "/"), -1)
clean_file_name = F.element_at(F.split(raw_file_name, "\\?"), 1)

df_with_meta = (
    df.withColumn("meta_load_timestamp", F.current_timestamp())
      .withColumn("meta_source_file", clean_file_name)
)

# Overwrite the managed Delta table
(
    df_with_meta.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Successfully loaded '{latest_file_path} ' into {target_table}")


StatementMeta(, 2bc70166-4f34-4162-ae65-e5a79a21336c, 17, Finished, Available, Finished, False)

Successfully loaded 'abfss://5e32bf37-c2be-4676-94c7-97213238ed30@onelake.dfs.fabric.microsoft.com/f27d086e-be7b-4e40-8e62-ea4d210b4538/Files/landing/oyster/20261109142245/Customer_20261109T142245.csv ' into dbo.ld_oys_customer


In [16]:
df = spark.sql(f"""SELECT 
    meta_load_timestamp,
    meta_source_file,
    COUNT(*) AS row_count
FROM {target_table}
GROUP BY meta_load_timestamp, meta_source_file
ORDER BY meta_source_file DESC""")

display(df)


StatementMeta(, 2bc70166-4f34-4162-ae65-e5a79a21336c, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 383e18da-aef1-4849-a5a5-eb6e7e4cc818)